# Task 9b — does the deletion change what the model *writes*, not only what it predicts?

**Run all cells.** Everything below is idempotent and resumable: re-running a cell is always safe,
and the generation phase picks up where it left off after a runtime disconnect.

The question. Task 9a's omittability result (far / close stratified ratio 1.85×) compares next-token
*distributions* after a splice. The paper's claim is about the *text* that follows. One greedy
continuation is not a usable measure: after a sentence boundary the next sentence is nearly free, so
greedy decodes from the original and the spliced context part within a median of 3.0 tokens for close
cuts and 1.5 for far ones, and that agreement length correlates with state divergence at only −0.21.
So this run uses **two text measures that do not depend on a single decode**, every parameter fixed
before anything is generated (they are the cache header, written before the first cut):

- **Measure A — the author's actual continuation.** For the `W_TRUE` tokens after the join, which
  are the same tokens in the original and the spliced sequence (`ids[j+1+k]` is `spliced[i+1+k]`),
  `true_dlogp = mean_k( log p(ids[j+1+k] | ids[:j+1+k]) − log p(spliced[i+1+k] | spliced[:i+1+k]) )`
  in nats per token; above 0 means the deletion made the real text less likely.
- **Measure B — sampled continuations, compared as distributions.** `K_SAMPLES` continuations of
  `L_NEW_TOKENS` tokens from context O = `ids[:j+1]` and from context S = `spliced[:i+1]` (the same
  token ends both; only the deleted sentences differ), nucleus sampling with `TOP_P` and `TEMPERATURE`,
  no top-k, no repetition penalty, EOS an ordinary token (never suppressed, never a stop), seeded per
  cut as `SEED_BASE + cut_index` with the **same seed for O and S** so the two sets are paired by random
  state. Token F1 over multisets of ids gives `within` (the O–O pairs), `cross` (the O–S pairs) and
  `sample_overlap = cross / within`; 1.0 means the spliced context writes the same distribution of text.
- **Descriptive only.** `first_diff_greedy`, the first position at which greedy decodes from O and S
  differ, capped at `L_NEW_TOKENS`. It is in the record because the theory document quotes it;
  nothing is judged on it.

Inputs are the 9a cache (paragraph text, per-token `surprisal`) and the 9a record (its `cuts` are
exactly the labelled close / far cuts, in order). **Neither is modified.** The output is a separate,
additive record, `record_9b_<RUN_TAG>.json`, with every sampled continuation stored so another overlap
function can be applied to it later, plus three invariants reported back: the cached surprisal
reproduces `lp_orig`; the cut keys match the 9a record exactly; `within > 0` for every cut. The
pre-registered analysis (runbook §3.9: far vs close on both measures within 9a's length strata,
Spearman with state divergence) is done from the record, outside this notebook.

What this notebook does:

1. Clones the private repo `SalmonSung/m1_llms_analyzer` and installs its dependencies.
2. Runs a **smoke test** of the whole chain on a ~5 MB model and the hand paragraphs.
3. Loads the model with its LM head and **restores the 9a cache and record from Drive** (read-only).
4. Walks through **one cut**: the join, both measures, the greedy pair.
5. **Phase A (GPU):** scores and samples every labelled cut into a resumable JSONL cache.
6. **Phase A′ (CPU):** builds the record, prints the three invariants, saves and persists it.


## 1 · Bootstrap — clone the repo

In [ ]:
#@title Clone (or update) the repository { display-mode: "form" }
import base64, json, os, subprocess, sys, textwrap, urllib.error, urllib.request
from pathlib import Path

REPO_OWNER  = "SalmonSung"
REPO_NAME   = "m1_llms_analyzer"
REPO_BRANCH = "main"   #@param {type:"string"}

CLEAN_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
NEW_PAT_URL = "https://github.com/settings/personal-access-tokens/new"


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


def _colab_secret(name):
    """Read a Colab secret, returning None if it is absent or access is denied."""
    if not _in_colab():
        return None
    try:
        from google.colab import userdata
        return userdata.get(name) or None
    except Exception as exc:
        print(f"  (Colab secret {name!r} unavailable: {type(exc).__name__})")
        return None


def _clean(token):
    """Strip whitespace and stray quotes -- by far the most common paste error."""
    if not token:
        return None
    return token.strip().strip('"').strip("'").strip() or None


def _token_kind(token):
    """Name the token type from its prefix, without revealing the value."""
    for prefix, kind in (
        ("github_pat_", "fine-grained PAT"),
        ("ghp_", "classic PAT"),
        ("gho_", "OAuth token"),
        ("ghs_", "App installation token"),
        ("ghu_", "user-to-server token"),
    ):
        if token.startswith(prefix):
            return kind
    return "UNRECOGNISED PREFIX"


def _api(path, token):
    """GET api.github.com/<path> with the token. Raises urllib.error.HTTPError."""
    request = urllib.request.Request(
        f"https://api.github.com/{path}",
        headers={
            "Authorization": f"Bearer {token}",
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
        },
    )
    with urllib.request.urlopen(request, timeout=30) as response:
        return json.loads(response.read().decode())


def _auth_config(token):
    """Auth as a per-command git config value, so it never touches .git/config or a URL."""
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    return f"http.extraHeader=AUTHORIZATION: basic {basic}"


def _run(cmd, token=None):
    """Run git, redacting the auth header and the token from anything printed."""
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        message = result.stderr or result.stdout
        if token:
            message = message.replace(token, "***")
        shown = " ".join("<auth>" if "extraHeader" in c else c for c in cmd)
        raise RuntimeError(f"git failed: {shown}\n{message}")
    return result.stdout.strip()


def _preflight(token):
    """Verify the token before git runs, so failures name their actual cause.

    A bare `git clone` failure says only 'Invalid username or token', which covers an
    expired token, a typo, a missing repo grant, and un-authorised SSO alike. These two
    API calls tell those apart.
    """
    kind = _token_kind(token)
    print(f"GITHUB_TOKEN: {len(token)} chars, looks like a {kind}.")
    if kind == "UNRECOGNISED PREFIX":
        print("  Warning: GitHub tokens start with github_pat_, ghp_, gho_, ghs_ or ghu_.")
        print("  If you pasted an account password or an SSH key, that will not work here.")

    try:
        me = _api("user", token)
    except urllib.error.HTTPError as exc:
        if exc.code == 401:
            raise SystemExit(textwrap.dedent(f"""
                GitHub rejected this token (401 Unauthorized). The token itself is bad --
                this is not a permissions problem. Most likely one of:

                  * it has expired (fine-grained PATs expire, 30 days by default);
                  * it was revoked or regenerated;
                  * the secret holds something that is not a token (an account password
                    will never work -- GitHub removed password auth for git);
                  * it was truncated or mangled when pasted.

                Fix: create a new token at
                  {NEW_PAT_URL}
                  - Resource owner: {REPO_OWNER}
                  - Repository access: only select repositories -> {REPO_NAME}
                  - Permissions: Repository permissions -> Contents -> Read-only
                Then in Colab: key icon in the left sidebar -> edit GITHUB_TOKEN, paste the
                new value with no quotes and no trailing spaces, keep 'Notebook access' on,
                and re-run this cell.
            """).strip())
        if exc.code == 403:
            raise SystemExit(textwrap.dedent(f"""
                GitHub returned 403 for this token. Usually either a rate limit, or the
                token needs SAML SSO authorisation for the '{REPO_OWNER}' organisation.
                If {REPO_OWNER} is an org with SSO, open your token's settings page and
                click 'Configure SSO' -> Authorize.

                Original error: {exc}
            """).strip())
        raise

    print(f"  Authenticates as: {me.get('login')}")

    try:
        repo = _api(f"repos/{REPO_OWNER}/{REPO_NAME}", token)
    except urllib.error.HTTPError as exc:
        if exc.code == 404:
            login = me.get("login")
            not_owner = (
                f"\n                  * you are {login}, but the repo belongs to "
                f"{REPO_OWNER} and you are not a collaborator on it;"
                if login and login.lower() != REPO_OWNER.lower()
                else ""
            )
            raise SystemExit(textwrap.dedent(f"""
                The token is valid (you are {login}), but it cannot see
                {REPO_OWNER}/{REPO_NAME}. GitHub returns 404 rather than 403 for a private
                repo a token has no grant on, so this means one of:

                  * the token's 'Repository access' does not include {REPO_NAME};
                  * it lacks the 'Contents: Read-only' repository permission;{not_owner}
                  * the owner or name is misspelled (both are case-sensitive).

                Fix: open {NEW_PAT_URL} (or edit the existing token), grant this
                repository and Contents: Read-only, then re-run this cell.
            """).strip())
        raise

    print(f"  Repo access:      OK ({'private' if repo.get('private') else 'public'})")
    return True


_TOKEN = _clean(_colab_secret("GITHUB_TOKEN") or os.environ.get("GITHUB_TOKEN"))

if not _in_colab() and Path("pyproject.toml").exists():
    # Running from a local checkout -- nothing to clone.
    REPO_DIR = Path.cwd()
    print(f"Local checkout detected: {REPO_DIR}")
else:
    REPO_DIR = Path("/content") / REPO_NAME if _in_colab() else Path.cwd() / REPO_NAME
    if not _TOKEN:
        raise SystemExit(textwrap.dedent(f"""
            GITHUB_TOKEN is not set, and {REPO_OWNER}/{REPO_NAME} is private.
              1. Create a fine-grained PAT at {NEW_PAT_URL}
                 - Resource owner: {REPO_OWNER}
                 - Repository access: only select repositories -> {REPO_NAME}
                 - Permissions: Contents -> Read-only
              2. Colab left sidebar -> key icon -> add a secret named GITHUB_TOKEN
              3. Turn on 'Notebook access' for it, then re-run this cell.
        """).strip())

    _preflight(_TOKEN)

    # Auth travels as a per-command header, never in the URL. Nothing is written to
    # .git/config, so there is no token left on disk to scrub afterwards.
    _AUTH = _auth_config(_TOKEN)
    try:
        if REPO_DIR.exists():
            print(f"\nRepo already present at {REPO_DIR}; updating...")
            _run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", CLEAN_URL], _TOKEN)
            _run(["git", "-C", str(REPO_DIR), "-c", _AUTH, "fetch", "origin", REPO_BRANCH], _TOKEN)
            _run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], _TOKEN)
            _run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"], _TOKEN)
        else:
            print(f"\nCloning into {REPO_DIR} ...")
            _run(["git", "-c", _AUTH, "clone", "--branch", REPO_BRANCH, "--depth", "1",
                  CLEAN_URL, str(REPO_DIR)], _TOKEN)
    except RuntimeError as exc:
        # The API accepted the token but git did not -- rare, and worth naming, because
        # the obvious readings (bad token, missing grant) were just ruled out above.
        raise SystemExit(textwrap.dedent(f"""
            {exc}

            The token passed the API preflight above, so it is valid and can see this
            repo -- the failure is in the git transport itself. Things to check:

              * branch '{REPO_BRANCH}' exists on the remote (a typo in REPO_BRANCH gives
                'Remote branch not found');
              * a stale {REPO_DIR} from an earlier run: delete it and re-run this cell;
              * a corporate proxy or VPN intercepting HTTPS to github.com.
        """).strip())
    print("Done. Remote is", CLEAN_URL, "(no credentials stored on disk).")

os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())
print("Commit:", _run(["git", "rev-parse", "--short", "HEAD"]))
del _TOKEN  # do not leave the token bound in the notebook namespace

## 2 · Dependencies

Installed with `--upgrade-strategy only-if-needed` so Colab's preinstalled,
CUDA-matched `torch` is **kept** rather than reinstalled (a torch swap costs several
minutes and can break GPU support).

In [ ]:
#@title Install dependencies
import subprocess, sys

print("Installing (quiet; ~30s on a cold runtime)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "--upgrade-strategy", "only-if-needed", "-e", "."],
    capture_output=True, text=True,
)
print(result.stdout[-2000:] or "(no output)")
if result.returncode != 0:
    print(result.stderr[-3000:], file=sys.stderr)
    raise SystemExit("Dependency installation failed -- see the error above.")

# Make the freshly installed package importable in this already-running kernel.
import importlib, site
importlib.reload(site)
for module in [m for m in list(sys.modules) if m.startswith("m1_analyzer")]:
    del sys.modules[module]

import m1_analyzer
print("m1_analyzer", m1_analyzer.__version__, "ready")

## 3 · Environment report

In [ ]:
#@title What am I running on?
import torch, transformers, numpy, platform
from m1_analyzer import in_colab, resolve_hf_token

print(f"python        : {platform.python_version()}")
print(f"torch         : {torch.__version__}")
print(f"transformers  : {transformers.__version__}")
print(f"numpy         : {numpy.__version__}")
print(f"in Colab      : {in_colab()}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU           : {props.name} ({props.total_memory / 1024**3:.1f} GB)")
    print(f"CUDA          : {torch.version.cuda}")
else:
    print("GPU           : none -- running on CPU.")
    print("                Runtime -> Change runtime type -> T4 GPU for anything above ~1B params.")

# Only reports presence. The token value is never printed.
print(f"HF_TOKEN      : {'found' if resolve_hf_token() else 'not set (fine for ungated models)'}")

# The dtype the scorer will run in. A T4 gets float16 (its bfloat16 is emulated and slow);
# Ampere and newer get bfloat16; CPU gets float32.
from m1_analyzer.utils.device import resolve_device, resolve_dtype
print(f"scoring dtype : {str(resolve_dtype(resolve_device('auto'), 'auto')).replace('torch.', '')}")

## 4 · Configuration

**This is the only cell you normally edit.** `MODEL_ID`, `CORPUS` and `WINDOW` must be the ones the
9a run was scored under: `RUN_TAG` is what names the 9a files this notebook reads and the 9b files it
writes. The protocol values are the pre-registration — they are displayed here so the record is
self-explaining, not offered for tuning: the generation cache header pins them, and a different
value refuses to resume an existing cache and names the field.


In [ ]:
#@title Run configuration { display-mode: "form" }
SMOKE_MODEL_ID  = "sshleifer/tiny-gpt2"        #@param {type:"string"}
MODEL_ID        = "Qwen/Qwen3-0.6B-Base"       #@param {type:"string"}
DTYPE           = "auto"                       #@param ["auto", "float16", "bfloat16", "float32"]
CORPUS          = "wikipedia"                  #@param {type:"string"}
WINDOW          = 20                           #@param {type:"integer"}
MAX_TOKENS      = 300                          #@param {type:"integer"}
# --- the pre-registered protocol (shown, not tuned) ---
W_TRUE          = 20                           #@param {type:"integer"}
K_SAMPLES       = 16                           #@param {type:"integer"}
L_NEW_TOKENS    = 30                           #@param {type:"integer"}
TOP_P           = 0.95                         #@param {type:"number"}
TEMPERATURE     = 1.0                          #@param {type:"number"}
SEED_BASE       = 20250914                     #@param {type:"integer"}
# --- run mechanics ---
USE_KV_CACHE    = True                         #@param {type:"boolean"}
LIMIT           = 0                            #@param {type:"integer"}
MIRROR_EVERY    = 10                           #@param {type:"integer"}
TOL_LP          = 1e-5                         #@param {type:"number"}
OUTPUT_DIR      = "outputs/task_9b"            #@param {type:"string"}
SEED            = 42                           #@param {type:"integer"}
RESUME          = True                         #@param {type:"boolean"}
MIRROR_TO_DRIVE = False                        #@param {type:"boolean"}
DRIVE_DIR_9A    = "/content/drive/MyDrive/m1_llms_analyzer/task_9a"  #@param {type:"string"}
DRIVE_DIR       = "/content/drive/MyDrive/m1_llms_analyzer/task_9b"  #@param {type:"string"}

import re
from pathlib import Path

from m1_analyzer import ModelConfig, RunConfig, ScoringConfig, StorageConfig
from m1_analyzer.experiments import GenerationProtocol

OUT = Path(OUTPUT_DIR)
OUT.mkdir(parents=True, exist_ok=True)
PROTOCOL = GenerationProtocol(W_true=W_TRUE, K=K_SAMPLES, L=L_NEW_TOKENS, top_p=TOP_P,
                              temperature=TEMPERATURE, seed_base=SEED_BASE, greedy_cap=L_NEW_TOKENS)


def slug(text: str) -> str:
    """Filesystem-safe name (model id, corpus) for output files."""
    return re.sub(r"-{2,}", "-", re.sub(r"[^A-Za-z0-9._-]+", "-", text)).strip("-")


def make_config(model_id: str, **overrides) -> RunConfig:
    """A RunConfig with the LM head, so next-token states, log-probabilities and sampling work."""
    return RunConfig(
        model=ModelConfig(model_id=model_id, head="causal_lm", dtype=overrides.get("dtype", DTYPE)),
        # bos_policy="auto" is what the 9a cache was scored under; the sampler prepends the same token.
        scoring=ScoringConfig(batch_size=16, max_length_cap=max(512, MAX_TOKENS + 8 + L_NEW_TOKENS)),
        storage=StorageConfig(output_dir=str(OUT)),
        seed=SEED,
    )


# The same RUN_TAG as Task 9a, so the 9a files are found and the 9b files sit beside them by name.
RUN_TAG = f"{slug(MODEL_ID)}_{slug(Path(CORPUS).stem if CORPUS != 'wikipedia' else 'wikipedia')}_w{WINDOW}"
CACHE_9A_PATH = OUT / f"splices_{RUN_TAG}.jsonl"      # read-only input
RECORD_9A_PATH = OUT / f"record_9a_{RUN_TAG}.json"    # read-only input
GEN_PATH = OUT / f"gen_9b_{RUN_TAG}.jsonl"            # phase A cache (resumable)
RECORD_PATH = OUT / f"record_9b_{RUN_TAG}.json"       # the deliverable

print(f"protocol : {PROTOCOL.as_header()['sampling']}; W_true {W_TRUE}, K {K_SAMPLES}, L {L_NEW_TOKENS}")
print(f"9a cache : {CACHE_9A_PATH}")
print(f"9a record: {RECORD_9A_PATH}")
print(f"9b cache : {GEN_PATH}")
print(f"9b record: {RECORD_PATH}")
print("Configuration ready.")


## 5 · Smoke test (~5 MB, a few seconds)

Runs the *entire* chain on a tiny model: a 9a cache and record on the hand paragraphs (the `regex`
splitter, a short window), then generation for every labelled cut with a scaled-down protocol, the
record, its validation and the three invariants. If this passes, the only things that can still go
wrong with the real model are download size, GPU memory, and the Drive paths.


In [ ]:
#@title Smoke test on a tiny model
import json
import time

from m1_analyzer import Analyzer
from m1_analyzer.experiments import (
    GenerationProtocol, Paragraph, analyse_9a, build_record_9b, compute_generation_9b, compute_splices,
    hand_paragraphs, validate_record_9b,
)

smoke = Analyzer(make_config(SMOKE_MODEL_ID))
SMOKE_DIR = OUT / "smoke"
SMOKE_DIR.mkdir(exist_ok=True)
for stale in SMOKE_DIR.glob("*"):
    stale.unlink()
print(smoke.describe()["head"], "|", smoke.models.architecture, "| BOS token:", smoke.states.bos_token())

started = time.perf_counter()
# Two copies of the hand paragraphs under different ids, so the length-matching bins hold enough
# cuts for a close and a far decile.
paragraphs = list(hand_paragraphs()) + [Paragraph(id=p.id + "-b", text=p.text, source=p.source, info=dict(p.info))
                                        for p in hand_paragraphs()]
header_9a, rows_9a = compute_splices(
    smoke.states, paragraphs, window=5, splitter="regex", cache_path=SMOKE_DIR / "splices.jsonl",
    strata=[[1, 10], [10, 30], [30, 80]], provenance={"model_id": SMOKE_MODEL_ID}, show_progress=False,
)
record_9a = analyse_9a(rows_9a, header_9a, model=SMOKE_MODEL_ID, n_perm=50, n_boot=20, min_per_group=1)
(SMOKE_DIR / "record_9a.json").write_text(json.dumps(record_9a), encoding="utf-8")
print(f"9a: {len(rows_9a)} paragraphs, {len(record_9a['cuts'])} labelled cuts")

smoke_protocol = GenerationProtocol(W_true=5, K=4, L=6, top_p=TOP_P, temperature=TEMPERATURE, seed_base=SEED_BASE, greedy_cap=6)
header, rows = compute_generation_9b(
    smoke.states, smoke.models, SMOKE_DIR / "record_9a.json", SMOKE_DIR / "splices.jsonl",
    cache_path=SMOKE_DIR / "gen_9b.jsonl", protocol=smoke_protocol, provenance={"model_id": SMOKE_MODEL_ID},
    show_progress=False,
)
smoke_record = build_record_9b(header, rows, record_9a, model=SMOKE_MODEL_ID)
validate_record_9b(smoke_record)
inv = smoke_record["invariants"]
print(f"9b: {len(rows)} cuts generated and assembled in {time.perf_counter() - started:.1f}s; "
      f"keys match {inv['keys_match']}, max |lp_orig + surprisal| = {inv['max_abs_cached_lp_diff']:.1e}, "
      f"collapsed {inv['n_collapsed']}")
assert inv["ok"], inv
print("\nSmoke test passed (the numbers above are meaningless: a random 5 MB model).")

smoke.unload()   # free the tiny model before loading the real one


## 6 · Load the model

First download takes a minute; it is cached for the rest of the session. The model is loaded
**with its language-model head** (`head="causal_lm"`): that is what scores a continuation and
samples one. Use the same `MODEL_ID` and dtype the 9a cache was scored under, so measure A's
original-side pass reproduces the cached surprisal (invariant 1 checks exactly that). If it is
gated, accept its licence on the Hub and add `HF_TOKEN` to Colab secrets.


In [ ]:
#@title Load the model
import time

from m1_analyzer import Analyzer

started = time.perf_counter()
analyzer = Analyzer(make_config(MODEL_ID))
print(f"Loaded in {time.perf_counter() - started:.1f}s\n")
for key, value in analyzer.describe().items():
    if key != "layers":
        print(f"{key:>18} : {value}")
print(f"{'BOS token':>18} : {analyzer.states.bos_token()!r} (prepended so position 0 has a state)")
print(f"{'vocabulary':>18} : {analyzer.states.vocab_size} (the dimension every distance lives in)")

count_tokens = lambda text: len(analyzer.states.encode(text))
PROVENANCE = {
    "model_id": MODEL_ID,
    "revision": analyzer.models.metadata()["revision"],
    "dtype": analyzer.models.metadata()["dtype"],
    "bos_token": analyzer.states.bos_token(),
    "vocab_size": analyzer.states.vocab_size,
    "seed": SEED,
}

## 7 · The 9a inputs (read-only)

The cache `splices_<RUN_TAG>.jsonl` and the record `record_9a_<RUN_TAG>.json` are copied from
`DRIVE_DIR_9A` when they are not on local disk (or when `LOAD_FROM_DRIVE` forces it), and then only
read: their sha256 goes into the generation cache header and the record, and the tests assert the
bytes never change. The record's `cuts` are the labelled cuts — the 428 in the reference run — in the
order that defines `cut_index`. The cell refuses to go on if the cache was scored with a different
model than this session loaded.


In [ ]:
#@title Restore the 9a cache and record from Drive (read-only)
import collections
import shutil

from m1_analyzer import in_colab
from m1_analyzer.experiments import load_9a_inputs

LOAD_FROM_DRIVE = False  #@param {type:"boolean"}

DRIVE_9A = Path(DRIVE_DIR_9A)
needed = [p for p in (CACHE_9A_PATH, RECORD_9A_PATH) if LOAD_FROM_DRIVE or not p.exists()]
if needed and in_colab() and not DRIVE_9A.exists():
    from google.colab import drive  # noqa: F401 -- Colab only
    drive.mount("/content/drive")
for path in needed:
    source = DRIVE_9A / path.name
    if not source.exists():
        raise SystemExit(
            f"Need {path.name}: not at {path} and not at {source}. Copy the Task 9a cache and record into "
            f"{DRIVE_DIR_9A} (the 9a notebook's Drive cell does that), or point DRIVE_DIR_9A at them. The names "
            "are keyed by MODEL_ID, CORPUS and WINDOW, which must match the 9a run.")
    path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(source, path)
    print(f"Restored {path.name} from Drive")

inputs = load_9a_inputs(RECORD_9A_PATH, CACHE_9A_PATH)
cuts_9a = inputs.record["cuts"]
print(f"9a record : {RECORD_9A_PATH.name}  sha256 {inputs.source_record['sha256'][:12]}  "
      f"model {inputs.source_record['model']}  date {inputs.source_record['date']}")
print(f"9a cache  : {CACHE_9A_PATH.name}  sha256 {inputs.source_cache['sha256'][:12]}  "
      f"{len(inputs.rows)} paragraphs, dtype {inputs.source_cache['dtype']}, bos {inputs.source_cache['bos_token']!r}")
print(f"labelled cuts: {len(cuts_9a)}  " + "  ".join(f"{k} {v}" for k, v in sorted(collections.Counter(c['pair'] for c in cuts_9a).items())))
for stratum, n in sorted(collections.Counter(c["stratum"] for c in cuts_9a).items()):
    print(f"   {stratum:<16} {n:>4} cuts")
print(f"paragraphs without `surprisal` (invariant 1 unavailable there): "
      f"{sum(1 for r in inputs.rows if 'surprisal' not in r)}")

cache_model = inputs.source_cache["model_id"]
if cache_model and cache_model != MODEL_ID:
    raise SystemExit(f"The 9a cache was scored with {cache_model!r}, this session loaded {MODEL_ID!r}. "
                     "Set MODEL_ID to the cache's model and re-run from the configuration cell.")
if inputs.source_cache["dtype"] and inputs.source_cache["dtype"] != PROVENANCE["dtype"]:
    print(f"NOTE: the 9a cache was scored in {inputs.source_cache['dtype']}, this session runs {PROVENANCE['dtype']}; "
          "invariant 1 may exceed the tolerance for that reason alone. It is reported, never adjusted.")


## 8 · One cut, end to end

The first labelled cut of the record: the join as text, measure A (the author's next `W_TRUE` tokens
under both contexts), measure B (`K_SAMPLES` paired samples from each context and their bag-of-token
overlap), and the greedy pair with its first difference. The seed is `SEED_BASE + cut_index`, so
re-running this cell reproduces the same samples on the same GPU.


In [ ]:
#@title Walk through one cut
from m1_analyzer.experiments import cut_specs, measure_a, measure_b

CUT_INDEX = 0  #@param {type:"integer"}

spec = cut_specs(inputs.record, inputs.rows)[CUT_INDEX]
states = analyzer.states
ids = states.encode(spec["text"])
assert len(ids) == spec["n_tokens"], "this is not the tokenizer the 9a cache was scored with"
i, j = spec["i"], spec["j"]
print(f"cut {spec['key']}  ({spec['pair']}, {spec['stratum']}, {spec['seg_len']} tokens deleted, state_div {spec['state_div']:.1f})")
print("   " + states.decode(ids[max(0, i - 12): i + 1]) + " ⟦cut⟧ " + states.decode(ids[j + 1: j + 13]) + "\n")

[orig] = states.states([ids], [[]], batch_size=1, return_token_logprobs=True)
a = measure_a(states, ids, i, j, PROTOCOL.W_true, orig_token_logprobs=orig.token_logprobs, cached_surprisal=spec["surprisal"])
print(f"measure A  true_dlogp = {a['true_dlogp']:+.4f} nats/token over {PROTOCOL.W_true} tokens; "
      f"max |lp_orig + cached surprisal| = {a['cached_lp_orig_max_abs_diff']:.1e}")
print("           per token: " + " ".join(f"{x:+.2f}" for x in a["true_dlogp_k"]))

max_length = analyzer.models.effective_max_length(states.config.max_length, states.config.max_length_cap)
b = measure_b(analyzer.models, ids, i, j, PROTOCOL, seed=PROTOCOL.seed_for(spec["cut_index"]), bos_id=states.bos_id(),
              kv_cache=USE_KV_CACHE, max_length=max_length)
print(f"measure B  within {b['within']:.3f}  cross {b['cross']:.3f}  sample_overlap {b['sample_overlap']}  "
      f"(EOS in {b['n_eos_orig']}/{PROTOCOL.K} original and {b['n_eos_spliced']}/{PROTOCOL.K} spliced samples)")
print(f"greedy     first_diff_greedy = {b['first_diff_greedy']}")
print("   O greedy :", repr(states.decode(b["greedy_orig"])))
print("   S greedy :", repr(states.decode(b["greedy_spliced"])))
print("   O sample0:", repr(states.decode(b["samples_orig"][0])))
print("   S sample0:", repr(states.decode(b["samples_spliced"][0])))


## 9 · Phase A — score and sample every labelled cut (GPU)

Per cut: one scored pass of the spliced ids (measure A; the original ids are scored once per
paragraph and shared), then `K_SAMPLES` samples of `L_NEW_TOKENS` from each context and one greedy
decode from each. One JSONL line per finished cut, fsynced, so a disconnect costs at most one cut:
re-run this cell and it resumes. The **header line is the pre-registration**: the protocol and the
sha256 of both 9a inputs; a run with a different protocol, or against a different 9a record or
cache, refuses to resume and names the field. `LIMIT > 0` scores only the first cuts (a dry run);
`MIRROR_TO_DRIVE` copies the cache to `DRIVE_DIR` every `MIRROR_EVERY` cuts. About 3 s per cut on a T4.


In [ ]:
#@title Generate for all labelled cuts (resumable)
import shutil
import time

from m1_analyzer.experiments import compute_generation_9b

if not RESUME and GEN_PATH.exists():
    GEN_PATH.unlink()
    print("RESUME=False: deleted the existing generation cache; starting from scratch.")

mirror = None
if MIRROR_TO_DRIVE:
    from google.colab import drive  # noqa: F401 -- Colab only
    drive.mount("/content/drive")
    mirror = Path(DRIVE_DIR) / GEN_PATH.name
    if not GEN_PATH.exists() and mirror.exists():
        shutil.copy(mirror, GEN_PATH)
        print(f"Restored the generation cache from Drive: {mirror}")

started = time.perf_counter()
header, rows = compute_generation_9b(
    analyzer.states, analyzer.models, RECORD_9A_PATH, CACHE_9A_PATH, cache_path=GEN_PATH, protocol=PROTOCOL,
    provenance=PROVENANCE, kv_cache=USE_KV_CACHE, show_progress=True, mirror_path=mirror,
    mirror_every=MIRROR_EVERY, limit=LIMIT or None,
)
elapsed = time.perf_counter() - started
print(f"\n{len(rows)}/{len(inputs.record['cuts'])} cuts in the cache ({elapsed / 60:.1f} min this run; "
      f"median {sorted(r['seconds'] for r in rows)[len(rows) // 2]:.2f} s per cut)")
print(f"  collapsed sample sets (within == 0): {sum(1 for r in rows if not r['within'] > 0)}")
print(f"  samples containing EOS: {sum(r['n_eos_orig'] for r in rows)} original, {sum(r['n_eos_spliced'] for r in rows)} spliced")


## 10 · Phase A′ — the record and the three invariants (CPU)

Reads the generation cache and the 9a record and assembles `record_9b_<RUN_TAG>.json`: per cut the
requested fields (`true_dlogp`, its 20 per-token values, `sample_overlap`, `within`, `cross`, all 32
continuations, `first_diff_greedy`, `state_div` copied from 9a) plus additive ones (`key`,
`cut_index`, `seed`, `grammatical`, EOS counts, the greedy pairs), the protocol, the meta, and the
invariants:

1. `-true_dlogp_k`'s original side, i.e. `lp_orig`, recomputed here, against the cached `surprisal`
   — max absolute difference (a warning above `TOL_LP`; nothing is adjusted);
2. the cut keys are exactly the 9a record's, in order — a mismatch stops the notebook;
3. `within > 0` for every cut — a collapsed sample set is listed; measure B is undefined there.

**This phase needs no GPU and no model** — only the generation cache and the 9a record, both restored
from Drive when `LOAD_FROM_DRIVE` is on or the local files are missing.


In [ ]:
#@title Build the record and report the invariants
import json
import shutil
from pathlib import Path

from m1_analyzer import in_colab
from m1_analyzer.experiments import build_record_9b, check_invariants_9b, load_generation_9b

LOAD_FROM_DRIVE = False  #@param {type:"boolean"}

DRIVE = Path(DRIVE_DIR)
if LOAD_FROM_DRIVE and in_colab() and not DRIVE.exists():
    from google.colab import drive  # noqa: F401 -- Colab only
    drive.mount("/content/drive")
if LOAD_FROM_DRIVE or not GEN_PATH.exists():
    if (DRIVE / GEN_PATH.name).exists():
        GEN_PATH.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(DRIVE / GEN_PATH.name, GEN_PATH)
        print(f"Restored the generation cache from Drive: {DRIVE / GEN_PATH.name}")
    if not GEN_PATH.exists():
        raise SystemExit(f"No generation cache at {GEN_PATH}, and none in {DRIVE}. Run Phase A, or point DRIVE_DIR at it.")
if not RECORD_9A_PATH.exists():
    raise SystemExit("Run '7 · The 9a inputs' first: the record needs the 9a record for the keys and state_div.")

header, rows = load_generation_9b(GEN_PATH)
record_9a = json.loads(RECORD_9A_PATH.read_text(encoding="utf-8"))
inv = check_invariants_9b(rows, record_9a, tol=TOL_LP)

n = inv["n_rows"]
lp = inv["max_abs_cached_lp_diff"]
print(f"invariant 1  lp_orig vs cached surprisal : max |diff| = {lp:.2e}  "
      f"({inv['n_cuts_within_tol']}/{n - inv['n_without_cached_surprisal']} cuts within {TOL_LP:g}"
      + (f"; {inv['n_without_cached_surprisal']} without cached surprisal)" if inv["n_without_cached_surprisal"] else ")")
      if lp is not None else "invariant 1  lp_orig vs cached surprisal : unavailable (the 9a cache has no surprisal field)")
print(f"invariant 2  keys match the 9a record    : {inv['keys_match'] and inv['order_ok']}  "
      f"({n}/{inv['n_expected']} cuts; missing {len(inv['missing'])}, extra {len(inv['extra'])}, order ok {inv['order_ok']})")
print(f"invariant 3  within > 0 for every cut     : {inv['n_collapsed'] == 0}  "
      f"({inv['n_collapsed']} collapsed" + (": " + ", ".join(inv["collapsed"][:5]) if inv["collapsed"] else "") + ")")
print(f"             samples with EOS: {inv['n_with_eos_orig']} cuts (original), {inv['n_with_eos_spliced']} cuts (spliced)")
if not (inv["keys_match"] and inv["order_ok"]):
    raise SystemExit("The generation cache does not cover the 9a record's cuts exactly. Finish Phase A (RESUME=True), "
                     "or check that RECORD_9A_PATH is the record this cache was generated from (its sha256 is in the header).")
if not inv["lp_orig_ok"]:
    print(f"WARNING: invariant 1 exceeds {TOL_LP:g}. A different dtype or GPU than the 9a run explains a small excess; "
          "the number is recorded as it is.")
if not inv["state_div_matches"]:
    raise SystemExit("state_div in the cache differs from the 9a record's divergence: wrong record file.")

record = build_record_9b(
    header, rows, record_9a, model=header.get("model_id", MODEL_ID), revision=header.get("revision"),
    dtype=header.get("dtype"), tol=TOL_LP,
    notes=f"corpus={CORPUS}; window={WINDOW}; kv_cache={header['protocol'].get('kv_cache')}; "
          f"9a record date={record_9a.get('meta', {}).get('date')}",
)
close = [c for c in record["cuts"] if c["pair"] == "close"]
far = [c for c in record["cuts"] if c["pair"] == "far"]
import numpy as np
def med(cs, key):
    vals = [c[key] for c in cs if c[key] is not None]
    return float(np.median(vals)) if vals else float("nan")
print(f"\n{'':<10}{'n':>6}{'true_dlogp':>12}{'sample_overlap':>16}{'first_diff':>12}   (medians; descriptive only, the analysis is done from the record)")
for name, cs in (("close", close), ("far", far)):
    print(f"{name:<10}{len(cs):>6}{med(cs, 'true_dlogp'):>12.4f}{med(cs, 'sample_overlap'):>16.4f}{med(cs, 'first_diff_greedy'):>12.1f}")


## 11 · Save and persist

The record is the experiment's single output. Colab wipes `/content` when the runtime is recycled,
so the Drive cell copies the record and the generation cache to `DRIVE_DIR` — never into the 9a
folder, whose files this notebook only reads.


In [ ]:
#@title Save the record
import json
import os
import tempfile

def atomic_write_json(path, payload):
    path = Path(path)
    fd, tmp = tempfile.mkstemp(dir=str(path.parent), prefix=f".{path.stem}.", suffix=".json")
    os.close(fd)
    Path(tmp).write_text(json.dumps(payload, indent=1, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, path)

atomic_write_json(RECORD_PATH, record)
print(f"record : {RECORD_PATH} ({RECORD_PATH.stat().st_size / 1024 / 1024:.1f} MB, {len(record['cuts'])} cuts)")
print(json.dumps({"protocol": record["protocol"], "meta": record["meta"],
                  "invariants": {k: v for k, v in record["invariants"].items() if k not in ("missing", "extra", "collapsed")}},
                 indent=2))


In [ ]:
#@title Persist results to Google Drive (survives a runtime disconnect)
import shutil

from m1_analyzer import in_colab

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    target = Path(DRIVE_DIR)
    target.mkdir(parents=True, exist_ok=True)
    for path in (RECORD_PATH, GEN_PATH):
        if path.exists():
            shutil.copy(path, target / path.name)
            print("copied", target / path.name)
else:
    print("Not running in Colab -- skipping Drive mount.")


In [ ]:
#@title Download the record to your machine
from m1_analyzer import in_colab

if in_colab():
    from google.colab import files
    if RECORD_PATH.exists():
        files.download(str(RECORD_PATH))
else:
    print("Not in Colab; the record is already on disk under", OUT)


In [ ]:
#@title Run the test suite inside Colab
import subprocess, sys

result_proc = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True
)
print(result_proc.stdout[-4000:])
print(result_proc.stderr[-2000:], file=sys.stderr)

---

## Troubleshooting

| Symptom | Fix |
|---|---|
| `GITHUB_TOKEN is not set` / `401` / `403` / `cannot see <repo>` | See the bootstrap cell's message: add or fix the fine-grained PAT (Contents: Read on this repo), enable *Notebook access*, re-run cell 1. |
| `Need splices_<RUN_TAG>.jsonl: not at ... and not at ...` | The 9a files are not where `DRIVE_DIR_9A` points. Run the 9a notebook's Drive cell, or copy its cache and record there; `MODEL_ID`, `CORPUS` and `WINDOW` must match the 9a run, since they name the files. |
| `The 9a cache was scored with ..., this session loaded ...` | Set `MODEL_ID` to the cache header's model and re-run from the configuration cell. |
| `the text re-encodes to N tokens but the 9a cache says M` | Not the tokenizer the cache was scored with (a different model or revision). Same fix. |
| `was written for protocol=... but this run has ...` | The generation cache belongs to a run with a different pre-registration. Change `OUTPUT_DIR`, or set `RESUME = False` to regenerate. |
| `was written for source_record=...` | The generation cache was built against a different 9a record or cache (their sha256 is in the header). Restore the matching 9a files, or start a new cache. |
| `non-finite logits while decoding` / `non-finite next-token state` | float16 overflowed inside the model. Set `DTYPE = "float32"`, `RESUME = False`, re-run Phase A (invariant 1 will then reflect the dtype change). |
| `CUDA out of memory` | The peak is the `K_SAMPLES` × context prefill. Use a bigger GPU or `DTYPE = "float16"`; the protocol's `K` is not a knob. `USE_KV_CACHE = False` trades speed for a little memory. |
| `Task 9b needs bos_policy='auto'` | The scoring config must prepend a BOS, as the 9a cache did; leave `ScoringConfig.bos_policy` at its default. |
| Invariant 1 above `TOL_LP` | A different dtype or GPU than the 9a run changes the low bits of the log-probabilities. The number is reported as it is; if it is large (> 1e-3), check `MODEL_ID` / revision against the cache header. |
| `The generation cache does not cover the 9a record's cuts exactly` | Phase A did not finish (re-run it with `RESUME = True`), or `LIMIT` was set for a dry run. |
| A fresh runtime, GPU gone | Phase A′ needs no model: run cells 1–4 (bootstrap and configuration), turn `LOAD_FROM_DRIVE` on in cell 7 and cell 10, and run those two cells. |
